# 19-Site Edge-Colored Candidate Scan

This notebook runs a coarse candidate scan over edge-colored HVA parameters. It is a smoke test, not a continuous VQE optimization. For the optimized pilot, use `19site_edge_colored_vqe.ipynb`.

In [ ]:
from pathlib import Path
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT = Path.cwd()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent

EXACT_ENERGY = float(np.load(PROJECT / "results" / "19site_fixed_sz_exact_n9.npz")["energy"])
plt.rcParams["figure.dpi"] = 130

## Run Experiment

In [ ]:
cmd = [
    sys.executable,
    str(PROJECT / "scripts" / "legacy" / "run_19site_edge_colored_hva.py"),
    "--max-layers",
    "4",
]
result = subprocess.run(cmd, cwd=PROJECT, text=True, capture_output=True, check=True)
print(result.stdout)

## Results Table

In [ ]:
df = pd.read_csv(PROJECT / "results" / "19site_edge_colored_candidate_scan.csv")
display_columns = [
    "ansatz",
    "depth",
    "energy",
    "error_vs_reference",
    "fidelity",
    "entropy",
    "max_magnetization",
]
df[display_columns]

## Energy and Fidelity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
plot_df = df[df["ansatz"].isin(["dimer_baseline", "shared_hva", "edge_colored_hva"])]

for ansatz, group in plot_df.groupby("ansatz"):
    axes[0].plot(group["depth"], group["energy"], marker="o", label=ansatz)
axes[0].axhline(EXACT_ENERGY, color="black", ls="--", lw=1, label="exact")
axes[0].set_xlabel("Depth p")
axes[0].set_ylabel("Energy")
axes[0].legend(fontsize=8)

for ansatz, group in plot_df.groupby("ansatz"):
    axes[1].plot(group["depth"], group["fidelity"], marker="o", label=ansatz)
axes[1].set_xlabel("Depth p")
axes[1].set_ylabel("Fidelity")

fig.tight_layout()
plt.show()